# 03 - Feature Engineering

**Author:** Juan Ruelas  
**Date:** 2026-05

We turn the cleaned hourly frame into a supervised-learning matrix. Four families of features are added, each motivated by what we saw in 02_eda:

1. **Calendar** - hour, day-of-week, month, weekend, German national holidays. These encode the human-behaviour drivers of demand.
2. **Lags** - 1, 2, 24, 48, 168 hours. Captures short-term persistence plus daily and weekly seasonality.
3. **Rolling stats** - 24h rolling mean and std. Smooths short-term noise and gives the model a sense of recent level and volatility.
4. **Interaction** - temperature x hour. Heating load is most temperature-sensitive in the evening; this term lets a tree model split on the combination directly.

Finally we split the frame chronologically: 2015-2018 -> train, 2019 -> test. Shuffling would leak future information through lag features, so we keep it strictly time-ordered.

In [ ]:
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

processed_dir = Path('..') / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(processed_dir / 'merged.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
print('input shape:', df.shape)
df.head()

## 1. Calendar features

We use Germany's national holiday calendar via the `holidays` package. Regional holidays (e.g. Reformation Day in some Lander) are intentionally excluded - they would inflate dimensionality without buying much because they affect only a fraction of national load.

In [ ]:
de_holidays = holidays.country_holidays('DE', years=range(2015, 2021))

df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['is_holiday'] = df['timestamp'].dt.date.map(lambda d: int(d in de_holidays))

df[['timestamp', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday']].head(8)

## 2. Lag features

Why these specific lags:

- **1h, 2h** - persistence; load barely changes minute-to-minute, so the previous hour is by far the strongest single predictor.
- **24h, 48h** - same hour yesterday and the day before. Captures daily seasonality robustly.
- **168h** - same hour same weekday a week ago. Strongest predictor that respects weekday-vs-weekend structure.

Lags introduce NaNs at the start of the series; we drop those rows once all features are built.

In [ ]:
for lag in [1, 2, 24, 48, 168]:
    df[f'load_lag_{lag}h'] = df['load_mw'].shift(lag)

## 3. Rolling statistics

We use `shift(1)` before rolling so that the value at time t only sees information up to t-1 - this avoids label leakage. A 24h window is a natural choice because it averages over one full daily cycle.

In [ ]:
shifted = df['load_mw'].shift(1)
df['load_roll_mean_24h'] = shifted.rolling(window=24, min_periods=24).mean()
df['load_roll_std_24h'] = shifted.rolling(window=24, min_periods=24).std()

## 4. Interaction term

Heating sensitivity to outdoor temperature varies by time of day. A single multiplicative term `temperature_2m * hour` is a cheap surrogate that lets a tree model split on the joint variable without needing many one-hot-by-temperature splits.

In [ ]:
df['temp_x_hour'] = df['temperature_2m'] * df['hour']

## 5. Drop NaNs from lags / rolls and check

In [ ]:
before = len(df)
df = df.dropna().reset_index(drop=True)
print(f'Dropped {before - len(df)} rows with NaN lag/rolling features')
print('Final shape:', df.shape)
df.head()

## 6. Time-based train / test split

- **Train**: 2015-01-08 (first valid row after 168h warm-up) -> 2018-12-31  
- **Test** : 2019-01-01 -> 2019-12-31

In [ ]:
train = df[df['timestamp'] < '2019-01-01'].copy()
test = df[df['timestamp'] >= '2019-01-01'].copy()

print('Train range:', train['timestamp'].min(), '->', train['timestamp'].max(), 'rows =', len(train))
print('Test  range:', test['timestamp'].min(), '->', test['timestamp'].max(), 'rows =', len(test))

In [ ]:
train_path = processed_dir / 'features_train.csv'
test_path  = processed_dir / 'features_test.csv'
train.to_csv(train_path, index=False)
test.to_csv(test_path, index=False)
print('saved', train_path)
print('saved', test_path)
print('feature cols:', [c for c in train.columns if c not in ('timestamp', 'load_mw')])

**Summary.** We now have 17 features in addition to the target. The train/test split is leak-free (chronological, lags built before split) and ready for modelling in notebook 04.